In [13]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from openpyxl import Workbook
from openpyxl.styles import PatternFill
from ipywidgets import FileUpload
from IPython.display import display
import io

In [14]:
# Create a file upload widget
upload = FileUpload(accept='.csv', multiple=False)  # Allow only CSV files
display(upload)  # Display the upload widget

# Wait for the user to upload a file
print("Please upload a CSV file.")

FileUpload(value=(), accept='.csv', description='Upload')

Please upload a CSV file.


In [15]:
# Crear un nuevo workbook de Excel
wb = Workbook()
output_file = "multi_sheet_defender_attacker_results_with_attacker_optimization.xlsx"

In [16]:
C = 1000  # Defender's total budget
A = 3  # Number of attacks the attacker can make
λ = 1  # Fixed lambda

In [17]:
# Función de probabilidad de éxito del ataque
def attack_success_prob(d):
    return np.exp(-λ * np.array(d))  # P_i(d_i) = e^(-λ * d_i)

# Función de probabilidad de éxito del defensor
def defender_success_prob(d):
    return 1 - attack_success_prob(d)

# Función objetivo del defensor (minimizar riesgo)
def defender_objective(d, a, v):
    P = attack_success_prob(d)
    return sum(a[i] * P[i] * v[i] for i in range(len(v)))

# Restricción del presupuesto del defensor
def budget_constraint(d):
    return C - np.sum(d)

# Función objetivo del atacante (maximizar riesgo)
def attacker_objective(a, d, v):
    P = attack_success_prob(d)
    return -sum(a[i] * P[i] * v[i] for i in range(len(v)))  # Negado para minimizar en lugar de maximizar

# Restricción para atacar exactamente A estaciones
def attack_constraint(a):
    return A - np.sum(a)


In [18]:
# Check if a file has been uploaded
if upload.value:
    # Get the uploaded file data
    uploaded_file = upload.value[0]  # Access the first uploaded file
    content = uploaded_file['content']  # Access the file's content
    
    # Read the CSV file from the uploaded content
    df = pd.read_csv(io.BytesIO(content))
    
    # Buscar la columna de atractivo
    score_column = next((col for col in df.columns if "Base_Attractiveness_Score" in col), None)
    if not score_column:
        raise ValueError("No 'Base_Attractiveness_Score' column found in the CSV file.")

    v = df[score_column].tolist()
    n = len(v)

    # Inicialización de variables
    d_init = np.random.uniform(0, C / n, n)
    a_init = np.random.uniform(0, 1, n)
    d_bounds = [(0, C/5) for _ in range(n)]
    constraints = {'type': 'eq', 'fun': budget_constraint}

    # Resolver la optimización del defensor
    d_result = minimize(defender_objective, d_init, args=(a_init, v), method='SLSQP', bounds=d_bounds, constraints=constraints)
    d_optimal = d_result.x

    # Resolver la optimización del atacante
    attacker_bounds = [(0, 1) for _ in range(n)]
    attacker_constraints = {'type': 'eq', 'fun': attack_constraint}
    a_result = minimize(attacker_objective, a_init, args=(d_optimal, v), method='SLSQP', bounds=attacker_bounds, constraints=attacker_constraints)
    a_optimal = a_result.x

    # Calcular la probabilidad de éxito del defensor
    P_defender = defender_success_prob(d_optimal)

    # Identificar las estaciones atacadas
    chosen_stations = np.argsort(a_optimal)[-A:]

    attack_status = ["Attack" if i in chosen_stations else "" for i in range(n)]

    # Crear el DataFrame de resultados
    result_df = pd.DataFrame({
        "Station Name": df["Station_Name"],
        "Allocated Defense Resources": d_optimal,
        "Attractiveness Score": v,
        "Attack Probability": a_optimal,
        "Defender Success Probability": P_defender,
        "Defender Budget": [C] * n,
        "Attack Status": attack_status
    })

    # Crear una nueva hoja en el archivo Excel
    ws = wb.create_sheet(title=uploaded_file['name'].split(".")[0][:30])

    # Escribir encabezados
    for col_idx, col_name in enumerate(result_df.columns, 1):
        ws.cell(row=1, column=col_idx, value=col_name)

    # Escribir datos en la hoja de Excel
    for r_idx, row in enumerate(result_df.itertuples(), 2):
        for c_idx, value in enumerate(row[1:], 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=value)

            # Colorear las filas de las estaciones atacadas
            if result_df.iloc[r_idx - 2]["Attack Status"] == "Attack":
                cell.fill = PatternFill(start_color="FF0000", end_color="FF0000", fill_type="solid")  # Rojo para ataque

    # Guardar el archivo Excel
    wb.save(output_file)
    print(f"Results saved to {output_file}")
else:
    print("No file uploaded.")

Results saved to multi_sheet_defender_attacker_results_with_attacker_optimization.xlsx
